# Hydra Houou + Throne BC Training on Colab

This notebook trains Hydra BC without Jade games. It uses only:

- Tenhou Houou MJAI
- MajSoul Throne MJAI

Upload these archives to `MyDrive/hydra/dataset/` before running:

- `hydra_houou_mjai.tar.zst`
- `hydra_throne_mjai.tar.zst`

All run artifacts go to Google Drive under `MyDrive/hydra/training/`. The launcher keeps checkpoints, writes JSONL logs, and can launch TensorBoard. Extra cells below expose TensorBoard through a temporary Cloudflare tunnel so it is reachable from your browser.

Use a GPU runtime: `Runtime -> Change runtime type -> GPU`.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ROOT = Path('/content/drive/MyDrive/hydra')
ARCHIVE_DIR = DRIVE_ROOT / 'dataset'
RUN_ROOT = DRIVE_ROOT / 'training'
BC_RUN_ROOT = RUN_ROOT / 'houou-throne-bc'
PPO_RUN_ROOT = RUN_ROOT / 'houou-throne-t1-ppo'
DATA_ROOT = Path('/content/hydra_data')
LOCAL_ARCHIVE_ROOT = Path('/content/hydra_archives')
REPO_ROOT = Path('/content/hydra')

HOUOU_ARCHIVE = ARCHIVE_DIR / 'hydra_houou_mjai.tar.zst'
THRONE_ARCHIVE = ARCHIVE_DIR / 'hydra_throne_mjai.tar.zst'

for path in (HOUOU_ARCHIVE, THRONE_ARCHIVE):
    if not path.exists():
        raise FileNotFoundError(f'Missing archive: {path}')

BC_RUN_ROOT.mkdir(parents=True, exist_ok=True)
PPO_RUN_ROOT.mkdir(parents=True, exist_ok=True)
DATA_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ARCHIVE_ROOT.mkdir(parents=True, exist_ok=True)
print('Drive root:', DRIVE_ROOT)
print('BC run root:', BC_RUN_ROOT)
print('PPO run root:', PPO_RUN_ROOT)
print('Data root:', DATA_ROOT)
print('Local archive root:', LOCAL_ARCHIVE_ROOT)
print('Houou archive GB:', round(HOUOU_ARCHIVE.stat().st_size / 1024**3, 2))
print('Throne archive GB:', round(THRONE_ARCHIVE.stat().st_size / 1024**3, 2))


## Install System Tools

Colab usually has `tar`, but `zstd` is installed explicitly for `.tar.zst` archives.


In [ ]:
!apt-get update -qq
__omp_shell("apt-get install -y -qq zstd curl git build-essential ca-certificates wget")


## Install Cloudflare Tunnel

This installs `cloudflared` so TensorBoard can be exposed with a temporary public URL. No Cloudflare account is required for quick tunnels.


In [ ]:
!bash -lc 'set -euxo pipefail; CLOUDFLARED=/content/cloudflared; if [ ! -x "$CLOUDFLARED" ]; then wget -q -O "$CLOUDFLARED" https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64; chmod 755 "$CLOUDFLARED"; fi; "$CLOUDFLARED" --version'


## Stage and Extract Houou + Throne Archives

Drive is persistent but slow for large sequential reads. This stages each archive onto local Colab disk with progress, then extracts locally with tar checkpoint output. Jade is excluded by construction.


In [ ]:
import shutil
import time

COPY_CHUNK_BYTES = 64 * 1024 * 1024
EXTRACT_LOG_INTERVAL_S = 10

def copy_with_progress(src: Path, dst: Path) -> None:
    if dst.exists() and dst.stat().st_size == src.stat().st_size:
        print(f'Local archive already staged: {dst.name}')
        return
    tmp = dst.with_suffix(dst.suffix + '.partial')
    total = src.stat().st_size
    copied = 0
    last_print = time.time()
    with src.open('rb') as fin, tmp.open('wb') as fout:
        while True:
            chunk = fin.read(COPY_CHUNK_BYTES)
            if not chunk:
                break
            fout.write(chunk)
            copied += len(chunk)
            now = time.time()
            if now - last_print >= 5 or copied == total:
                print(f'{dst.name}: copy {copied / 1024**3:.2f}/{total / 1024**3:.2f} GiB ({copied / total:.1%})', flush=True)
                last_print = now
    tmp.replace(dst)


def data_root_size_bytes() -> int:
    total = 0
    for path in DATA_ROOT.iterdir():
        if path.is_dir() and (path.name.startswith('tenhou-houou-mjai-') or path.name.startswith('majsoul-throne-mjai-')):
            try:
                total += path.stat().st_size
            except FileNotFoundError:
                pass
    return total


def extract_with_progress(local_archive: Path) -> None:
    before_bytes = data_root_size_bytes()
    start = time.time()
    last_bytes = before_bytes
    last_time = start
    print(f'{local_archive.name}: extract start', flush=True)
    proc = subprocess.Popen([
        'tar', '--use-compress-program=zstd -d -T0',
        '-xf', str(local_archive), '-C', str(DATA_ROOT),
    ])
    while True:
        time.sleep(EXTRACT_LOG_INTERVAL_S)
        now = time.time()
        current_bytes = data_root_size_bytes()
        added = max(0, current_bytes - before_bytes)
        delta = max(0, current_bytes - last_bytes)
        mb_s = delta / max(now - last_time, 1e-9) / 1024**2
        elapsed_min = (now - start) / 60
        print(f'{local_archive.name}: extract +{added / 1024**3:.2f} GiB, {mb_s:.1f} MiB/s recent, {elapsed_min:.1f} min', flush=True)
        last_bytes = current_bytes
        last_time = now
        returncode = proc.poll()
        if returncode is not None:
            if returncode != 0:
                raise subprocess.CalledProcessError(returncode, proc.args)
            break
    print(f'{local_archive.name}: extract done', flush=True)


def extract_once(archive: Path, marker: Path) -> None:
    if marker.exists():
        print(f'Skip already extracted: {archive.name}')
        return
    local_archive = LOCAL_ARCHIVE_ROOT / archive.name
    copy_with_progress(archive, local_archive)
    extract_with_progress(local_archive)
    marker.write_text('ok\n')

extract_once(HOUOU_ARCHIVE, DATA_ROOT / '.houou_extract_done')
extract_once(THRONE_ARCHIVE, DATA_ROOT / '.throne_extract_done')
print('done')


In [ ]:
houou_dirs = sorted(DATA_ROOT.glob('tenhou-houou-mjai-*'))
throne_dirs = sorted(DATA_ROOT.glob('majsoul-throne-mjai-*'))
if not houou_dirs or not throne_dirs:
    raise RuntimeError('Expected extracted Houou and Throne directories')
print('Houou dirs:', len(houou_dirs))
print('Throne dirs:', len(throne_dirs))
print('First Houou:', houou_dirs[0])
print('First Throne:', throne_dirs[0])


## Clone Hydra and Install Pixi


In [ ]:
!bash -lc 'set -euxo pipefail; if [ ! -d /content/hydra ]; then git clone --depth 1 --filter=blob:none --sparse https://github.com/NikkeTryHard/hydra.git /content/hydra; git -C /content/hydra sparse-checkout set Cargo.toml Cargo.lock pyproject.toml pixi.lock example.yaml LICENSE README.md assets crates python scripts; else git -C /content/hydra pull --ff-only; fi'


In [ ]:
import os

os.chdir(REPO_ROOT)
print('repo:', Path.cwd())


In [ ]:
from pathlib import Path
import shutil

pixi = shutil.which('pixi') or str(Path.home() / '.pixi' / 'bin' / 'pixi')
print(f'Pixi candidate: {pixi}', flush=True)


In [ ]:
%%bash
set -euxo pipefail
if [ ! -x "$HOME/.pixi/bin/pixi" ] && ! command -v pixi >/dev/null 2>&1; then
  echo 'Installing Pixi...'
  curl -fsSL https://pixi.sh/install.sh | sh
else
  echo 'Pixi already installed.'
fi


In [ ]:
from pathlib import Path
import shutil

pixi = shutil.which('pixi') or str(Path.home() / '.pixi' / 'bin' / 'pixi')
print(f'Pixi path: {pixi}', flush=True)


In [ ]:
!bash -lc 'set -euxo pipefail; export PATH="$HOME/.pixi/bin:$PATH"; pixi --version; echo "Running first Pixi command: pixi install"; cd /content/hydra; pixi install; echo "pixi install complete"'


## Build Raw MJAI PyO3 Extension

Pinned raw-MJAI transport uses a Rust/PyO3 shared library. Build it once before training.


In [ ]:
!bash -lc 'set -euxo pipefail; export PATH="$HOME/.pixi/bin:$PATH"; cd /content/hydra; echo "Building raw MJAI PyO3 extension..."; pixi run cargo build -p hydra-raw-mjai-pyo3 --release --quiet; test -f /content/hydra/target/release/libhydra_raw_mjai_pyo3.so'


In [ ]:
raw_mjai_pyo3_lib = REPO_ROOT / 'target' / 'release' / 'libhydra_raw_mjai_pyo3.so'
if not raw_mjai_pyo3_lib.exists():
    raise FileNotFoundError(raw_mjai_pyo3_lib)
print('raw MJAI PyO3:', raw_mjai_pyo3_lib)


## Write Houou + Throne BC Config

This config keeps the large BC profile and writes logs/checkpoints to Google Drive.


In [ ]:
import yaml

config = {
    'data_dir': str(DATA_ROOT),
    'raw_mjai_data_dirs': [str(p) for p in houou_dirs + throne_dirs],
    'output_dir': str(BC_RUN_ROOT),
    'stage': 'T0_bc_houou_throne',
    'run_name': 'latest_run',
    'num_epochs': 1,
    'full_epoch': True,
    'max_train_steps': None,
    'python_model_profile': 'large',
    'python_backbone_profile': 'conv2d_local3',
    'python_residual_profile': 'mish_se',
    'python_variant': 'compile_max_autotune',
    'python_conv_memory_format': 'contiguous',
    'batch_size': 3072,
    'microbatch_size': 1024,
    'validation_microbatch_size': 1024,
    'device': 'cuda:0',
    'bc_backend': 'python',
    'python_raw_mjai_transport': 'pinned_pyo3',
    'python_raw_mjai_target_games': 3330000,
    'train_fraction': 0.9,
    'augment': True,
    'seed': 0,
    'source_filters': {},
    'resume_checkpoint': None,
    'resume_latest': True,
    'log_every_n_steps': 50,
    'validate_every_n_steps': 1000,
    'checkpoint_every_n_steps': 2500,
    'keep_step_checkpoints': True,
    'validation_every_n_epochs': 1,
    'max_validation_batches': None,
    'max_validation_samples': 524288,
    'max_skip_logs_per_source': 32,
    'tensorboard': True,
    'launch_tensorboard': True,
    'tensorboard_host': '127.0.0.1',
    'tensorboard_port': 6006,
    'background': True,
    'shard_prefetch_depth': 2,
    'bc': {
        'learning_rate': 0.0002,
        'min_learning_rate': 0.000001,
        'weight_decay': 0.00001,
        'grad_clip_norm': 1.0,
        'warmup_steps': 1000,
        'adamw_fused': 'on',
        'adamw_foreach': 'auto',
    },
    'ema': {
        'enabled': True,
        'decay': 0.999,
        'start_step': 0,
        'update_every_steps': 1,
        'device': 'auto',
    },
    'advanced_loss': None,
    'exit_sidecar_path': None,
    'delta_q_sidecar_path': None,
    'validation_gates': {
        'enabled': False,
        'min_validation_samples': 1024,
        'max_policy_loss_regression': 0.0,
        'min_policy_agreement_delta': None,
        'fail_training_on_gate_failure': False,
        'require_sidecar_coverage_when_weighted': True,
    },
    'rl': None,
}

config_path = REPO_ROOT / 'colab_houou_throne_bc.yaml'
config_path.write_text(yaml.safe_dump(config, sort_keys=False))
print(config_path)
print(config_path.read_text()[:2000])


## Start BC Training

The launcher runs in background, prints the TensorBoard URL, and writes logs/checkpoints under the Drive run root.


In [ ]:
!bash -lc 'set -euxo pipefail; export PATH="$HOME/.pixi/bin:$PATH"; export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; export HYDRA_RAW_MJAI_PYO3_LIB=/content/hydra/target/release/libhydra_raw_mjai_pyo3.so; cd /content/hydra; pixi run cargo run --quiet --package hydra-train --features training --bin train -- /content/hydra/colab_houou_throne_bc.yaml'


## Watch BC Logs and Checkpoints

BC logs and checkpoints are on Google Drive. `latest.pt` is the resumable checkpoint; `step_<global_step>.pt` files are retained by config.


In [ ]:
bc_run_dir = BC_RUN_ROOT / 'stages' / 'T0_bc_houou_throne' / 'runs' / 'latest_run'
print('BC run dir:', bc_run_dir)
print('logs:', bc_run_dir / 'logs')
print('checkpoints:', bc_run_dir / 'checkpoints')
print('train pid file:', bc_run_dir / 'train.pid')


In [ ]:
!tail -n 40 "$bc_run_dir/logs/train_steps.jsonl"


In [ ]:
if (bc_run_dir / 'checkpoints').exists():
    for path in sorted((bc_run_dir / 'checkpoints').iterdir()):
        if path.is_file():
            print(f'{path.name} {path.stat().st_size} bytes')
else:
    print('No checkpoint directory yet')


## TensorBoard Through Cloudflare

Run this after training has created the TensorBoard/log directory. It starts TensorBoard and prints a public `trycloudflare.com` URL.


In [ ]:
import re
import time

_tunnel_processes = []

def start_tensorboard_tunnel(logdir: Path, port: int = 6006) -> str:
    if not logdir.exists():
        raise FileNotFoundError(f'Missing TensorBoard logdir: {logdir}')
    tb = subprocess.Popen([
        pixi, 'run', 'tensorboard',
        '--logdir', str(logdir),
        '--host', '0.0.0.0',
        '--port', str(port),
    ], cwd=REPO_ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    _tunnel_processes.append(tb)
    time.sleep(5)
    tunnel = subprocess.Popen([
        str(CLOUDFLARED), 'tunnel', '--url', f'http://127.0.0.1:{port}', '--no-autoupdate'
    ], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    _tunnel_processes.append(tunnel)
    deadline = time.time() + 60
    seen = []
    while time.time() < deadline:
        line = tunnel.stdout.readline() if tunnel.stdout else ''
        if line:
            seen.append(line.rstrip())
            match = re.search(r'https://[-a-zA-Z0-9.]+\.trycloudflare\.com', line)
            if match:
                url = match.group(0)
                print('TensorBoard public URL:', url)
                return url
        time.sleep(0.2)
    raise RuntimeError('Cloudflare tunnel URL not found. Last output:' + chr(10) + chr(10).join(seen[-20:]))

bc_tb_dir = bc_run_dir / 'tensorboard'
bc_tensorboard_url = start_tensorboard_tunnel(bc_tb_dir, 6006)


## Optional T1 PPO Config

Run this only after BC has produced `checkpoints/latest.pt`. It seeds T1 PPO from that BC checkpoint, keeps outputs in Google Drive, uses MahJAX GPU rollout, and enables MahJAX AOT.

These numbers are intentionally easy to tune later for Colab GPU performance.


In [ ]:
bc_checkpoint = bc_run_dir / 'checkpoints' / 'latest.pt'
if not bc_checkpoint.exists():
    raise FileNotFoundError(f'BC checkpoint not found yet: {bc_checkpoint}')

ppo_config = dict(config)
ppo_config.update({
    'output_dir': str(PPO_RUN_ROOT),
    'stage': 'T1_ppo_control',
    'run_name': 'latest_run',
    'resume_checkpoint': str(bc_checkpoint),
    'resume_latest': False,
    'num_epochs': 1,
    'full_epoch': False,
    'max_train_steps': None,
    'checkpoint_every_n_steps': 250,
    'keep_step_checkpoints': True,
    'tensorboard': True,
    'launch_tensorboard': True,
    'background': True,
    'rl': {
        'phase': 'ppo_control',
        'run_forever': True,
        'games_per_batch': 512,
        'microbatch_size': 768,
        'temperature': 1.0,
        'learning_rate': 0.0001,
        'min_learning_rate': 0.000001,
        'rollout_inference': 'mahjax-gpu',
        'ppo_pipeline_depth': 1,
        'bc_kl_reverse_coef': 0.0,
        'lr_warmup_samples': 0,
        'lr_decay_samples': None,
        'target_kl': 0.005,
    },
})

ppo_config_path = REPO_ROOT / 'colab_houou_throne_t1_ppo.yaml'
ppo_config_path.write_text(yaml.safe_dump(ppo_config, sort_keys=False))
print(ppo_config_path)
print(ppo_config_path.read_text()[:2400])


## Start Optional T1 PPO

This launches PPO in background. It uses the same Drive-backed run layout and prints local TensorBoard info from the launcher.


In [ ]:
!bash -lc 'set -euxo pipefail; export PATH="$HOME/.pixi/bin:$PATH"; export HYDRA_MAHJAX_AOT=1; export PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True; export XLA_PYTHON_CLIENT_PREALLOCATE=false; export XLA_PYTHON_CLIENT_MEM_FRACTION=0.45; cd /content/hydra; pixi run cargo run --quiet --package hydra-train --features training --bin train -- /content/hydra/colab_houou_throne_t1_ppo.yaml'


## Watch Optional T1 PPO


In [ ]:
ppo_run_dir = PPO_RUN_ROOT / 'stages' / 'T1_ppo_control' / 'runs' / 'latest_run'
print('PPO run dir:', ppo_run_dir)
print('logs:', ppo_run_dir / 'logs')
print('checkpoints:', ppo_run_dir / 'checkpoints')
print('train pid file:', ppo_run_dir / 'train.pid')


In [ ]:
!tail -n 40 "$ppo_run_dir/logs/train_steps.jsonl"


In [ ]:
if (ppo_run_dir / 'checkpoints').exists():
    for path in sorted((ppo_run_dir / 'checkpoints').iterdir()):
        if path.is_file():
            print(f'{path.name} {path.stat().st_size} bytes')
else:
    print('No checkpoint directory yet')


In [ ]:
ppo_tensorboard_url = start_tensorboard_tunnel(ppo_run_dir / 'tensorboard', 6007)
